# DGS Binary and Regression Models

This notebook introduces the standard scalar prediction workflows in DeepGeSeq (DGS): binary classification and regression. It builds directly on the data and loader concepts from notebook 0.


## Tutorial series map

This notebook is part of the `Tutorials/0_DGS_usage_models` sequence:
- `0_DGS_data.ipynb`: data, IO, intervals, targets, and loaders
- **`1_DGS_models.ipynb`: binary classification and regression scalar models**
- `2_DGS_profile_models.ipynb`: profile/count and long-sequence track models
- `3_DGS_gLMs.ipynb`: genomic language model adapters and downstream heads
- `4_DGS_explain.ipynb`: interpretation methods, attribution artifacts, and motif discovery

The notebooks are designed to be read in order, but each one keeps its examples self-contained and guarded against missing optional dependencies.


## Audience, prerequisites, and learning goals

This notebook is for users who want a practical starting point for DGS models before moving to profile-based models or genomic language model adapters.

Prerequisites:

- Basic Python and PyTorch concepts.
- One-hot DNA encoding.
- A high-level idea of supervised learning.

By the end, you should be able to:

- Choose the correct DGS task type for binary classification or regression.
- Instantiate a lightweight `CNN` model for scalar outputs.
- Match model outputs with an appropriate loss function.
- Compute classification and regression metrics with DGS evaluator helpers.
- Adapt the synthetic examples to real FASTA, BED, and target-track data.


## Outline

1. Set up a safe tutorial environment.
2. Inspect scalar model metadata in the DGS model zoo.
3. Create a small synthetic DNA dataset.
4. Train and evaluate a binary classification model.
5. Train and evaluate a regression model.
6. Compare binary vs regression design choices.
7. Connect the examples to real DGS dataloaders and configuration files.
8. Review exercises, pitfalls, and extensions.


## Series-level conventions

Across these notebooks:

- Core examples use tiny synthetic data or local fixtures and avoid model downloads.
- Optional real-data or external-checkpoint cells are disabled with `RUN_*` flags until you edit paths and opt in.
- Loader sequence batches are usually `(batch, length, 4)`; CNN-style models usually expect `(batch, 4, length)` after `transpose(1, 2)`.
- Scalar targets use `(batch, tasks)`; profile targets use `(batch, tracks, profile_length)`.
- Environment guards print skip messages when optional packages such as `torch`, `pysam`, `pyBigWig`, `transformers`, or `evo2` are unavailable.


## 1. Environment setup

The examples use synthetic tensors so they do not require any external genomic files. If `torch` or the local DGS package is unavailable in the current kernel, execution cells will print a clear skip message instead of failing.


In [ ]:
from pathlib import Path
import os
import sys

try:
    from IPython.display import display
except Exception:
    def display(obj):
        print(obj)


def find_repo_root(start):
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / ".git").exists() and (candidate / "DGS").exists():
            return candidate
    return start

REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

try:
    import numpy as np
    import pandas as pd
    import torch
    TORCH_AVAILABLE = True
except Exception as exc:
    np = None
    pd = None
    torch = None
    TORCH_AVAILABLE = False
    TORCH_IMPORT_ERROR = repr(exc)

DGS_READY = False
if TORCH_AVAILABLE:
    try:
        from DGS.Model import CNN, CAN, DeepSEA, get_model_zoo
        from DGS.DL.Evaluator import calculate_classification_metrics, calculate_regression_metrics
        DGS_READY = True
    except Exception as exc:
        DGS_IMPORT_ERROR = repr(exc)

print(f"Repository root: {REPO_ROOT}")
print(f"Torch available: {TORCH_AVAILABLE}")
if not TORCH_AVAILABLE:
    print(f"Torch import error: {TORCH_IMPORT_ERROR}")
print(f"DGS scalar model API ready: {DGS_READY}")
if TORCH_AVAILABLE and not DGS_READY:
    print(f"DGS import error: {DGS_IMPORT_ERROR}")


## 2. Scalar models in the DGS model zoo

For most first-pass scalar tasks, start with `CNN`.

- `CNN`: recommended lightweight baseline for binary and regression workflows.
- `CAN`: CNN plus attention blocks; useful after a baseline is working.
- `DeepSEA`, `DanQ`, `Basset`, and related publication models: useful when you need those specific architectures, but they are not the simplest tutorial entry point.

The key parameter is `output_size`, the number of scalar targets per sequence. For a single binary label or one regression value, use `output_size=1`.


In [ ]:
if DGS_READY:
    model_zoo = get_model_zoo()
    for name in ["CNN", "CAN", "DeepSEA", "DanQ", "Basset"]:
        if name not in model_zoo:
            continue
        meta = model_zoo[name]
        print(f"{name}")
        print(f"  input_shape:  {meta['input_shape']}")
        print(f"  output_shape: {meta['output_shape']}")
        print(f"  task_type:    {meta['task_type']}")
        print(f"  status:       {meta['workflow_status']}")
        print()
else:
    print("Skipped: DGS scalar model API is not ready in this kernel.")


## 3. Create a synthetic DNA dataset

We will generate one-hot DNA sequences with shape `(batch, 4, sequence_length)`, which is the standard channel-first shape used by DGS convolutional sequence models.

The synthetic labels are intentionally simple:

- Binary label: whether a short motif-like pattern is present.
- Regression target: a continuous activity score based on motif count and GC content.

This toy setup is not biological evidence; it is a compact way to learn the model interface.


In [ ]:
if DGS_READY:
    torch.manual_seed(12)
    np.random.seed(12)

    n_samples = 128
    sequence_length = 100
    train_size = 96
    motif = torch.tensor([0, 1, 2, 3, 0])  # A, C, G, T, A in index form.
    motif_length = len(motif)

    base_indices = torch.randint(0, 4, (n_samples, sequence_length))
    has_motif = torch.zeros(n_samples, dtype=torch.float32)
    motif_counts = torch.zeros(n_samples, dtype=torch.float32)

    for row in range(n_samples):
        if row % 2 == 0:
            start = torch.randint(0, sequence_length - motif_length + 1, (1,)).item()
            base_indices[row, start:start + motif_length] = motif
            has_motif[row] = 1.0
            motif_counts[row] += 1.0
        if row % 11 == 0:
            start = torch.randint(0, sequence_length - motif_length + 1, (1,)).item()
            base_indices[row, start:start + motif_length] = motif
            motif_counts[row] += 1.0

    one_hot_nlc = torch.nn.functional.one_hot(base_indices, num_classes=4).float()
    sequences_ncl = one_hot_nlc.transpose(1, 2).contiguous()

    gc_fraction = one_hot_nlc[:, :, [1, 2]].sum(dim=(1, 2)) / sequence_length
    binary_targets = has_motif.unsqueeze(1)
    regression_targets = (0.7 * motif_counts + 1.5 * gc_fraction + 0.05 * torch.randn(n_samples)).unsqueeze(1)

    train_sequences = sequences_ncl[:train_size]
    test_sequences = sequences_ncl[train_size:]
    train_binary = binary_targets[:train_size]
    test_binary = binary_targets[train_size:]
    train_regression = regression_targets[:train_size]
    test_regression = regression_targets[train_size:]

    print("sequences_ncl:", tuple(sequences_ncl.shape), "= (samples, 4, length)")
    print("binary_targets:", tuple(binary_targets.shape), "positive rate=", float(binary_targets.mean()))
    print("regression_targets:", tuple(regression_targets.shape), "range=", (float(regression_targets.min()), float(regression_targets.max())))
else:
    print("Skipped: DGS scalar model API is not ready in this kernel.")


In [ ]:
if DGS_READY:
    def count_parameters(model):
        return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)

    def make_batches(x, y, batch_size=32, shuffle=True):
        indices = torch.arange(x.shape[0])
        if shuffle:
            indices = indices[torch.randperm(len(indices))]
        for start in range(0, len(indices), batch_size):
            batch_idx = indices[start:start + batch_size]
            yield x[batch_idx], y[batch_idx]

    def train_for_few_epochs(model, criterion, optimizer, x, y, epochs=5, batch_size=32):
        history = []
        model.train()
        for epoch in range(epochs):
            epoch_losses = []
            for xb, yb in make_batches(x, y, batch_size=batch_size, shuffle=True):
                optimizer.zero_grad()
                pred = model(xb)
                loss = criterion(pred, yb)
                loss.backward()
                optimizer.step()
                epoch_losses.append(float(loss.detach()))
            history.append(sum(epoch_losses) / len(epoch_losses))
        return history

    def as_numpy(tensor):
        return tensor.detach().cpu().numpy()
else:
    print("Skipped: helper functions require DGS_READY.")


## 4. Binary classification model

For DGS `CNN`, use `tasktype="binary_classification"` when you want probability-like outputs from a sigmoid predictor.

Because the model already applies a sigmoid in this task mode, use `torch.nn.BCELoss()` with the native `CNN` binary output. If you build a custom model that returns raw logits instead, use `torch.nn.BCEWithLogitsLoss()` and apply `torch.sigmoid()` only for metrics.


In [ ]:
if DGS_READY:
    binary_model = CNN(
        output_size=1,
        out_planes=32,
        kernel_size=7,
        tasktype="binary_classification",
    )
    binary_criterion = torch.nn.BCELoss()
    binary_optimizer = torch.optim.Adam(binary_model.parameters(), lr=1e-3)

    binary_history = train_for_few_epochs(
        binary_model,
        binary_criterion,
        binary_optimizer,
        train_sequences,
        train_binary,
        epochs=6,
        batch_size=32,
    )

    print(f"Binary CNN trainable parameters: {count_parameters(binary_model):,}")
    print("training loss history:", [round(value, 4) for value in binary_history])
else:
    print("Skipped: DGS scalar model API is not ready in this kernel.")


In [ ]:
if DGS_READY:
    binary_model.eval()
    with torch.no_grad():
        binary_pred = binary_model(test_sequences)
        binary_loss = binary_criterion(binary_pred, test_binary)

    binary_metrics = calculate_classification_metrics(
        as_numpy(test_binary),
        as_numpy(binary_pred),
        threshold=0.5,
    )

    print(f"test BCELoss: {binary_loss.item():.4f}")
    print("prediction range:", (float(binary_pred.min()), float(binary_pred.max())))
    display(binary_metrics)
else:
    print("Skipped: DGS scalar model API is not ready in this kernel.")


### Interpreting the binary example

The model output is a probability-like score between 0 and 1. A threshold such as 0.5 converts scores into hard labels for F1 and accuracy.

For imbalanced genomics datasets, AUROC can look strong even when precision is poor. Always inspect AUPRC and task-specific thresholds when positives are rare.


## 5. Regression model

For continuous targets, use `tasktype="regression"`. The DGS predictor uses an identity output in this mode, so predictions are unconstrained real values.

Typical loss choices:

- `torch.nn.MSELoss()` for ordinary least-squares style training.
- `torch.nn.L1Loss()` or `torch.nn.SmoothL1Loss()` when outliers are important.
- A transformed target, such as `log1p(signal)`, when the raw target is highly skewed.


In [ ]:
if DGS_READY:
    regression_model = CNN(
        output_size=1,
        out_planes=32,
        kernel_size=7,
        tasktype="regression",
    )
    regression_criterion = torch.nn.MSELoss()
    regression_optimizer = torch.optim.Adam(regression_model.parameters(), lr=1e-3)

    regression_history = train_for_few_epochs(
        regression_model,
        regression_criterion,
        regression_optimizer,
        train_sequences,
        train_regression,
        epochs=6,
        batch_size=32,
    )

    print(f"Regression CNN trainable parameters: {count_parameters(regression_model):,}")
    print("training loss history:", [round(value, 4) for value in regression_history])
else:
    print("Skipped: DGS scalar model API is not ready in this kernel.")


In [ ]:
if DGS_READY:
    regression_model.eval()
    with torch.no_grad():
        regression_pred = regression_model(test_sequences)
        regression_loss = regression_criterion(regression_pred, test_regression)

    regression_metrics = calculate_regression_metrics(
        as_numpy(test_regression),
        as_numpy(regression_pred),
    )

    print(f"test MSELoss: {regression_loss.item():.4f}")
    print("target range:", (float(test_regression.min()), float(test_regression.max())))
    print("prediction range:", (float(regression_pred.min()), float(regression_pred.max())))
    display(regression_metrics)
else:
    print("Skipped: DGS scalar model API is not ready in this kernel.")


### Interpreting the regression example

Regression metrics answer different questions:

- `mse`, `rmse`, and `mae` measure prediction error in target units.
- `r2` measures variance explained.
- `pearson_r` measures linear ranking agreement.
- `spearman_r` is often useful when relative ordering matters more than exact scale.

In real MPRA-like tasks, report both error metrics and correlations.


## 6. Binary vs regression: what changes?

| Design choice | Binary classification | Regression |
|---|---|---|
| DGS `tasktype` | `binary_classification` | `regression` |
| Output meaning | probability-like score | continuous value |
| Typical DGS `CNN` loss | `BCELoss` | `MSELoss` |
| Common metrics | AUROC, AUPRC, F1, accuracy | MSE, MAE, R2, Pearson, Spearman |
| Target shape | `(batch, n_tasks)` with 0/1 values | `(batch, n_tasks)` with real values |

For multi-task binary or regression problems, set `output_size` to the number of targets and keep the target tensor shape aligned with the model output.


In [ ]:
if DGS_READY:
    multitask_binary = CNN(output_size=3, out_planes=16, kernel_size=5, tasktype="binary_classification")
    multitask_regression = CNN(output_size=3, out_planes=16, kernel_size=5, tasktype="regression")

    with torch.no_grad():
        binary_three_task_output = multitask_binary(test_sequences[:4])
        regression_three_task_output = multitask_regression(test_sequences[:4])

    print("multi-task binary output:", tuple(binary_three_task_output.shape))
    print("multi-task binary range:", (float(binary_three_task_output.min()), float(binary_three_task_output.max())))
    print("multi-task regression output:", tuple(regression_three_task_output.shape))
    print("multi-task regression range:", (float(regression_three_task_output.min()), float(regression_three_task_output.max())))
else:
    print("Skipped: DGS scalar model API is not ready in this kernel.")


## 7. Replace synthetic tensors with real DGS dataloaders

In a real project, you usually build training examples from FASTA sequences, BED intervals, and one or more target tracks. For scalar tasks, use supervised dataloaders with fixed target definitions.

The cell below is a template. It is disabled by default because it requires local data files.


In [ ]:
RUN_REAL_DATALOADER_EXAMPLE = False

if RUN_REAL_DATALOADER_EXAMPLE:
    from DGS.Data import build_supervised_dataloaders

    fasta_path = "path/to/genome.fa"
    intervals_path = "path/to/windows.bed"
    target_tasks = [
        {
            "task_name": "enhancer_label",
            "file_path": "path/to/enhancer_labels.bed",
            "file_type": "bed",
            "task_type": "binary",
        }
    ]

    train_loader, val_loader, test_loader = build_supervised_dataloaders(
        fasta_path=fasta_path,
        intervals_path=intervals_path,
        target_tasks=target_tasks,
        batch_size=32,
        split="random",
        random_state=12,
    )

    model = CNN(output_size=len(target_tasks), tasktype="binary_classification")
    criterion = torch.nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    sequences, targets = next(iter(train_loader))
    if sequences.shape[-1] == 4:
        sequences = sequences.transpose(1, 2).contiguous()
    predictions = model(sequences)
    loss = criterion(predictions, targets.float())
    print(loss)
else:
    print("Real dataloader example is disabled. Set RUN_REAL_DATALOADER_EXAMPLE = True after editing file paths.")


## 8. Configuration-style sketch

DGS command-line workflows can express the same choices in configuration files. The important pieces are the model output size and criterion.

Binary sketch:

```json
{
  "model": {"type": "CNN", "args": {"output_size": 1, "tasktype": "binary_classification"}},
  "train": {"criterion": {"type": "BCELoss", "params": {}}}
}
```

Regression sketch:

```json
{
  "model": {"type": "CNN", "args": {"output_size": 1, "tasktype": "regression"}},
  "train": {"criterion": {"type": "MSELoss", "params": {}}}
}
```

If your custom model returns logits for binary targets, use `BCEWithLogitsLoss` instead of `BCELoss` and do not add a sigmoid layer inside the model.


## 9. Exercise

Modify the binary example into a two-task binary classifier.

Suggested second label:

- Task 1: motif present.
- Task 2: GC fraction greater than 0.5.

Checklist:

- Build a target tensor with shape `(samples, 2)`.
- Instantiate `CNN(output_size=2, tasktype="binary_classification")`.
- Use `torch.nn.BCELoss()`.
- Evaluate each task with `calculate_classification_metrics`.


In [ ]:
if DGS_READY:
    gc_label = (gc_fraction > 0.5).float().unsqueeze(1)
    two_task_targets = torch.cat([binary_targets, gc_label], dim=1)

    two_task_model = CNN(output_size=2, out_planes=16, kernel_size=5, tasktype="binary_classification")
    two_task_criterion = torch.nn.BCELoss()
    two_task_optimizer = torch.optim.Adam(two_task_model.parameters(), lr=1e-3)

    two_task_history = train_for_few_epochs(
        two_task_model,
        two_task_criterion,
        two_task_optimizer,
        sequences_ncl[:train_size],
        two_task_targets[:train_size],
        epochs=3,
        batch_size=32,
    )

    two_task_model.eval()
    with torch.no_grad():
        two_task_pred = two_task_model(test_sequences)

    print("two-task target shape:", tuple(two_task_targets.shape))
    print("two-task prediction shape:", tuple(two_task_pred.shape))
    print("training loss history:", [round(value, 4) for value in two_task_history])
    display(calculate_classification_metrics(as_numpy(two_task_targets[train_size:]), as_numpy(two_task_pred)))
else:
    print("Skipped: DGS scalar model API is not ready in this kernel.")


## Common pitfalls

Pitfall 1: using the wrong binary loss.

- DGS `CNN(tasktype="binary_classification")` returns sigmoid probabilities, so pair it with `BCELoss`.
- A custom logits model should use `BCEWithLogitsLoss` instead.

Pitfall 2: target shape mismatch.

- For `output_size=1`, targets should usually be shaped `(batch, 1)`, not just `(batch,)`.
- For multi-task outputs, use `(batch, n_tasks)`.

Pitfall 3: treating regression outputs as probabilities.

- Regression predictions are unconstrained values.
- Apply target transformations deliberately and invert them when interpreting predictions.

Pitfall 4: relying on accuracy for imbalanced binary genomics data.

- Accuracy can be misleading when positives are rare.
- Prefer AUROC, AUPRC, and threshold-specific precision/recall checks.


## Optional extensions

- Swap `CNN` for `CAN` after the baseline works.
- Train for more epochs and plot loss curves.
- Replace synthetic labels with real BED or table-based targets.
- Try multi-task regression with several MPRA activity measurements.
- Add held-out chromosome splits for a more realistic generalization test.
